## retrieval QA benchmark

In [1]:
import sys
from pathlib import Path

# -----------------------------------------------------
# ADD PROJECT ROOT TO PYTHON PATH
# -----------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root added:")
print(PROJECT_ROOT)

# -----------------------------------------------------
# IMPORTS
# -----------------------------------------------------

from pprint import pprint

from src.retrieval.retriever import (
    retrieve_chunks
)

from src.structured.retrieval import (
    get_complete_site_profile,
    summarize_site_profile
)

print("\nImports successful")

Project root added:
D:\sk\planso_assignment

Imports successful


In [2]:
TEST_CASES = [

    {
        "case_id": 1,

        "bbl": "4049630075",

        "question": (
            "Does this site have an "
            "E designation and what "
            "does that mean?"
        ),

        "expected_sections": [

            "09_ceqr_e_designations",

        ],

        "expected_keywords": [

            "hazardous materials",
            "environmental requirement",
            "permit",
            "remediation"
        ]
    },

    {
        "case_id": 2,

        "bbl": "4049630075",

        "question": (
            "What zoning district governs "
            "this property and what "
            "bulk regulations apply?"
        ),

        "expected_sections": [

            "23-22",
            "23-43"

        ],

        "expected_keywords": [

            "C4-2",
            "floor area ratio",
            "height",
            "setback"
        ]
    },

    {
        "case_id": 3,

        "bbl": "2037690057",

        "question": (
            "What rear yard regulations "
            "apply to this site?"
        ),

        "expected_sections": [

            "23-342",
            "23-341"

        ],

        "expected_keywords": [

            "rear yard",
            "obstruction",
            "minimum depth"
        ]
    },

    {
        "case_id": 4,

        "bbl": "5004980028",

        "question": (
            "Does the special district "
            "affect zoning controls?"
        ),

        "expected_sections": [

            "23-344",

        ],

        "expected_keywords": [

            "special district",
            "alternative regulations",
            "supersede"
        ]
    },

    {
        "case_id": 5,

        "bbl": "4049630075",

        "question": (
            "Can HVAC equipment project "
            "into a required rear yard?"
        ),

        "expected_sections": [

            "23-341",
            "23-311"

        ],

        "expected_keywords": [

            "HVAC",
            "air conditioning",
            "rear yard",
            "obstruction"
        ]
    }
]

print(f"Loaded {len(TEST_CASES)} test cases")



Loaded 5 test cases


In [3]:
results = []

for case in TEST_CASES:

    print("\n" + "=" * 100)

    print(f"CASE {case['case_id']}")

    print("=" * 100)

    print("\nBBL:")
    print(case["bbl"])

    print("\nQUESTION:")
    print(case["question"])

    # -------------------------------------------------
    # STRUCTURED PROFILE
    # -------------------------------------------------

    profile = get_complete_site_profile(
        case["bbl"]
    )

    summary = summarize_site_profile(
        profile
    )

    print("\nSITE SUMMARY:\n")

    pprint(summary)

    # -------------------------------------------------
    # LEGAL RETRIEVAL
    # -------------------------------------------------

    retrieved_chunks, warnings = retrieve_chunks(

        question=case["question"],

        top_k=5,

        threshold=0.38
    )

    print("\nRETRIEVED CHUNKS:\n")

    for idx, chunk in enumerate(
        retrieved_chunks,
        start=1
    ):

        print(
            f"{idx}. "
            f"{chunk['source_file']} "
            f":: "
            f"{chunk['subsection_title']}"
        )

        print(
            f"   method="
            f"{chunk['retrieval_method']}"
        )

    print("\nWARNINGS:\n")

    if warnings:

        for w in warnings:
            print("-", w)

    else:

        print("None")

    results.append({

        "case": case,

        "summary": summary,

        "chunks": retrieved_chunks,

        "warnings": warnings
    })



CASE 1

BBL:
4049630075

QUESTION:
Does this site have an E designation and what does that mean?
[INFO] Loading site records from:
D:\sk\planso_assignment\corpus\structured\site_records.csv

[INFO] Loaded 5 site records
[INFO] Loading PLUTO from:
D:\sk\planso_assignment\corpus\structured\pluto_25v4.csv

[INFO] Loaded 858644 PLUTO records

SITE SUMMARY:

{'address': '37-52 College Point Blvd',
 'bbl': '4049630075',
 'borough': 'Queens',
 'building_area_sqft': '13440',
 'e_designation': 'Y',
 'e_designation_type': 'Hazardous Materials',
 'flood_zone': '',
 'lot_area_sqft': '29700',
 'notes': 'Commercial corridor; 1-story retail building; C4-2 district within '
          'Special Flushing Waterfront District (FW); (E) hazardous materials '
          'designation present — investigation required before permit',
 'overlay': '',
 'special_district': 'FW',
 'year_built': '1957',
 'zoning_district': 'C4-2'}
[INFO] Connecting to ChromaDB:
D:/sk/planso_assignment/chroma_db

[INFO] Available col

In [4]:
evaluation_rows = []

for item in results:

    case = item["case"]

    retrieved_text = " ".join([

        (
            c["subsection_title"]
            + " "
            + c["text"]
        )

        for c in item["chunks"]

    ]).lower()

    # =================================================
    # SECTION MATCH
    # =================================================

    section_hit = False

    matched_sections = []

    for sec in case["expected_sections"]:

        if sec.lower() in retrieved_text:

            section_hit = True

            matched_sections.append(sec)

    # =================================================
    # KEYWORD MATCH
    # =================================================

    matched_keywords = []

    for kw in case["expected_keywords"]:

        if kw.lower() in retrieved_text:

            matched_keywords.append(kw)

    keyword_score = round(

        len(matched_keywords)
        / len(case["expected_keywords"]),

        2
    )

    # =================================================
    # STORE
    # =================================================

    evaluation_rows.append({

        "case_id": case["case_id"],

        "question": case["question"],

        "section_hit": section_hit,

        "matched_sections": matched_sections,

        "keyword_score": keyword_score,

        "matched_keywords": matched_keywords,

        "warnings_count": len(
            item["warnings"]
        )
    })

print("\nEVALUATION RESULTS\n")

for row in evaluation_rows:

    print("=" * 80)

    pprint(row)


EVALUATION RESULTS

{'case_id': 1,
 'keyword_score': 1.0,
 'matched_keywords': ['hazardous materials',
                      'environmental requirement',
                      'permit',
                      'remediation'],
 'matched_sections': ['09_ceqr_e_designations'],
 'question': 'Does this site have an E designation and what does that mean?',
 'section_hit': True,
 'warnings_count': 0}
{'case_id': 2,
 'keyword_score': 0.0,
 'matched_keywords': [],
 'matched_sections': [],
 'question': 'What zoning district governs this property and what bulk '
             'regulations apply?',
 'section_hit': False,
 'warnings_count': 2}
{'case_id': 3,
 'keyword_score': 1.0,
 'matched_keywords': ['rear yard', 'obstruction', 'minimum depth'],
 'matched_sections': ['23-342', '23-341'],
 'question': 'What rear yard regulations apply to this site?',
 'section_hit': True,
 'warnings_count': 4}
{'case_id': 4,
 'keyword_score': 0.33,
 'matched_keywords': ['supersede'],
 'matched_sections': ['23-344'],

In [5]:
total_cases = len(
    evaluation_rows
)

section_hits = sum(

    r["section_hit"]

    for r in evaluation_rows
)

avg_keyword_score = round(

    sum(

        r["keyword_score"]

        for r in evaluation_rows

    ) / total_cases,

    2
)

avg_warning_count = round(

    sum(

        r["warnings_count"]

        for r in evaluation_rows

    ) / total_cases,

    2
)

print("\n" + "=" * 80)

print("FINAL RETRIEVAL METRICS")

print("=" * 80)

print(
    f"\nTotal Test Cases: "
    f"{total_cases}"
)

print(
    f"\nSection Hit Rate: "
    f"{round(section_hits / total_cases, 2)}"
)

print(
    f"\nAverage Keyword Score: "
    f"{avg_keyword_score}"
)

print(
    f"\nAverage Warning Count: "
    f"{avg_warning_count}"
)


FINAL RETRIEVAL METRICS

Total Test Cases: 5

Section Hit Rate: 0.8

Average Keyword Score: 0.62

Average Warning Count: 3.2


In [6]:
print("\nRETRIEVAL FAILURE ANALYSIS\n")

for row in evaluation_rows:

    print("=" * 80)

    print(f"CASE {row['case_id']}")

    print("-" * 80)

    print("QUESTION:")

    print(row["question"])

    print("\nSECTION HIT:")

    print(row["section_hit"])

    print("\nKEYWORD SCORE:")

    print(row["keyword_score"])

    print("\nWARNINGS:")

    print(row["warnings_count"])

    # -------------------------------------------------
    # DIAGNOSIS
    # -------------------------------------------------

    if not row["section_hit"]:

        print(
            "\nFAILURE TYPE: "
            "expected section not retrieved"
        )

    elif row["keyword_score"] < 0.5:

        print(
            "\nFAILURE TYPE: "
            "semantic mismatch"
        )

    elif row["warnings_count"] > 3:

        print(
            "\nFAILURE TYPE: "
            "dependency expansion too noisy"
        )

    else:

        print(
            "\nSTATUS: "
            "good retrieval"
        )


RETRIEVAL FAILURE ANALYSIS

CASE 1
--------------------------------------------------------------------------------
QUESTION:
Does this site have an E designation and what does that mean?

SECTION HIT:
True

KEYWORD SCORE:
1.0

WARNINGS:
0

STATUS: good retrieval
CASE 2
--------------------------------------------------------------------------------
QUESTION:
What zoning district governs this property and what bulk regulations apply?

SECTION HIT:
False

KEYWORD SCORE:
0.0

WARNINGS:
2

FAILURE TYPE: expected section not retrieved
CASE 3
--------------------------------------------------------------------------------
QUESTION:
What rear yard regulations apply to this site?

SECTION HIT:
True

KEYWORD SCORE:
1.0

WARNINGS:
4

FAILURE TYPE: dependency expansion too noisy
CASE 4
--------------------------------------------------------------------------------
QUESTION:
Does the special district affect zoning controls?

SECTION HIT:
True

KEYWORD SCORE:
0.33

WARNINGS:
4

FAILURE TYPE: sem